# Sprint 6: GraphRAG Optimizer Loop (LLM Integration)

## Purpose
Implement Epic 4 - GraphRAG optimizer loop with LLM integration:
- Serialize causal subgraph → structured LLM prompt
- Generate NL optimizer suggestions via LLM API
- Validate suggestions with what-if analysis
- Compute Recall@1 for bottleneck identification

## Acceptance Criteria
- Prompt template finalized, fits context window, node IDs traceable
- Ranked suggestions for ≥20 graphs
- ≥2 of 3 top suggestions reduce predicted ΔT
- Recall@1 computed across ≥20 graphs

## 1. Setup & Configuration

In [12]:
import os 
import json 
import torch
import numpy as np 
import pandas as pd 
import networkx as nx 
from pathlib import Path 
import torch_geometric.data  
from dotenv import load_dotenv
import matplotlib.pyplot as plt 
from typing import Dict, List, Tuple
from llama_index.llms.cohere import Cohere
from llama_index.llms.openai import OpenAI

In [4]:
load_dotenv()
api_key = os.environ["COHERE"]

## Loading Graphs to Memory

In [13]:
## IMPORTANT MAKESURE TO iNCULCATE THE GNN
import networkx as nx
import torch

def hetero_to_nx(hetero_data):
    """
    Convert HeteroData to a NetworkX DiGraph with node attributes:
    - 'type' : 'compute' or 'comm'
    - 'time' : execution time (float)
    Also adds edges from the HeteroData edge_index.
    """
    G = nx.DiGraph()
    
    # Add compute nodes
    if 'compute' in hetero_data.node_types:
        comp_x = hetero_data['compute'].x  # shape (N_comp, F)
        for i in range(comp_x.shape[0]):
            node_id = f"compute_{i}"
            G.add_node(node_id, type='compute', time=comp_x[i, 0].item())
    
    # Add comm nodes
    if 'comm' in hetero_data.node_types:
        comm_x = hetero_data['comm'].x
        for i in range(comm_x.shape[0]):
            node_id = f"comm_{i}"
            G.add_node(node_id, type='comm', time=comm_x[i, 0].item())
    
    # Add edges (assuming only one edge type, e.g., ('compute','to','comm') or ('comm','to','compute'))
    # HeteroData stores edge_index per edge type.
    for edge_type in hetero_data.edge_types:
        src_type, _, dst_type = edge_type
        edge_index = hetero_data[edge_type].edge_index
        for src_local, dst_local in edge_index.t().tolist():
            src_node = f"{src_type}_{src_local}"
            dst_node = f"{dst_type}_{dst_local}"
            G.add_edge(src_node, dst_node)
    
    return G

def compute_slack_cpm(G):
    """
    Compute earliest start, latest start, and slack for each node.
    Assumes node attribute 'time' (duration).
    Returns dict {node_id: slack}.
    """
    # Topological order (DAG assumed)
    topo_order = list(nx.topological_sort(G))
    
    # Earliest start time (EST)
    est = {node: 0.0 for node in G.nodes}
    for node in topo_order:
        for pred in G.predecessors(node):
            est[node] = max(est[node], est[pred] + G.nodes[pred]['time'])
    
    # Project duration
    project_duration = max(est[node] + G.nodes[node]['time'] for node in G.nodes)
    
    # Latest start time (LST)
    lst = {node: project_duration - G.nodes[node]['time'] for node in G.nodes}
    for node in reversed(topo_order):
        for succ in G.successors(node):
            lst[node] = min(lst[node], lst[succ] - G.nodes[node]['time'])
    
    # Slack = LST - EST
    slack = {node: lst[node] - est[node] for node in G.nodes}
    return slack

In [ ]:
def load_graphs_from_directory(graph_dir, map_location='cpu'):
    """
    Load all .pt graph files from directory.
    Uses weights_only=False (trusted source) and maps to CPU.
    """
    graphs = []
    graph_files = list(Path(graph_dir).glob("**/*.pt"))
    
    print(f"Found {len(graph_files)} .pt files. Loading...")
    
    for graph_file in graph_files:
        try:
            # Critical fixes:
            # - weights_only=False  (allows HeteroData)
            # - map_location='cpu'  (handles GPU->CPU transfer)
            graph = torch.load(graph_file, weights_only=False, map_location=map_location)
            graphs.append(graph)
        except Exception as e:
            print(f"Error loading {graph_file}: {e}")
    
    print(f"Successfully loaded {len(graphs)} graphs")
    return graphs

# Usage
graph_path = "./data/graphs"
all_graphs = load_graphs_from_directory(graph_path)

if all_graphs:
    sample_graph = all_graphs[0]
    print(f"\nSample graph structure:")
    print(f"Node types: {sample_graph.node_types}")
    print(f"Edge types: {sample_graph.edge_types}")
    for node_type in sample_graph.node_types:
        print(f"{node_type}: {sample_graph[node_type].num_nodes} nodes")
        if hasattr(sample_graph[node_type], 'x'):
            print(f"  Features: {sample_graph[node_type].x.shape}")

C:\Users\Aaron\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Found 503 .pt files. Loading...
Successfully loaded 503 graphs

Sample graph structure:
Node types: ['compute']
Edge types: [('compute', 'depends_on', 'compute')]
compute: 1486 nodes
  Features: torch.Size([1486, 4])


## 2. Graph Serialization & Prompt Engineering

TODO: Develop prompt template for graph-to-text conversion

In [15]:
def iter_hetero_nodes(hetero_data):
    for node_type in hetero_data.node_types:
        num_nodes = hetero_data[node_type].num_nodes
        for i in range(num_nodes):
            yield f"{node_type}_{i}", node_type
            
def graph_to_prompt(hetero_data, slack_dict):
    """
    Convert a PyG HeteroData graph and its slack predictions into an LLM prompt.
    """
    prompt = (
        "You are an expert AI performance engineer. "
        "Analyze the following computation graph and its critical path slack scores.\n"
        "Slack lower means more critical bottleneck.\n\n"
        "Node ID | Node Type | Slack\n"
        "--------|-----------|------\n"
    )
    for node_id, node_type in iter_hetero_nodes(hetero_data):
        slack = slack_dict.get(node_id, float('inf'))
        prompt += f"{node_id} | {node_type} | {slack:.4f}\n"
    prompt += (
        "\nBased on the slack scores, identify the top 3 bottleneck nodes and suggest "
        "specific optimizations (e.g., increase microbatches, overlap compute/communication, "
        "remap stages to different GPUs). Return the answer as JSON with keys: "
        "'top_bottlenecks' (list of node IDs), 'suggestions' (list of strings)."
    )
    return prompt

In [17]:
# # Cell 5: Generate and print the prompt for inspection
# sample_prompt = graph_to_prompt(graph, slack_scores)
# print(sample_prompt[:1000])   # first 1000 chars

# Load one graph (already loaded as `sample_graph`)
G = hetero_to_nx(sample_graph)
slack_dict = compute_slack_cpm(G)

# Now use your existing graph_to_prompt function (the one that iterates over HeteroData nodes)
prompt = graph_to_prompt(sample_graph, slack_dict)
print(prompt)

You are an expert AI performance engineer. Analyze the following computation graph and its critical path slack scores.
Slack lower means more critical bottleneck.

Node ID | Node Type | Slack
--------|-----------|------
compute_0 | compute | 0.0000
compute_1 | compute | 0.0219
compute_2 | compute | 0.0000
compute_3 | compute | 0.0380
compute_4 | compute | 0.0433
compute_5 | compute | 0.0466
compute_6 | compute | 0.0380
compute_7 | compute | 0.0627
compute_8 | compute | 0.0798
compute_9 | compute | 0.0893
compute_10 | compute | 0.0929
compute_11 | compute | 0.0893
compute_12 | compute | 0.0798
compute_13 | compute | 0.0627
compute_14 | compute | 0.0627
compute_15 | compute | 0.0380
compute_16 | compute | 0.0000
compute_17 | compute | 2.5770
compute_18 | compute | 0.0000
compute_19 | compute | 2.5906
compute_20 | compute | 0.0000
compute_21 | compute | 2.5969
compute_22 | compute | 0.0000
compute_23 | compute | 2.6054
compute_24 | compute | 0.0000
compute_25 | compute | 2.6092
compute_26

## 3. LLM Integration

TODO: Connect to LLM API and generate optimization suggestions

In [25]:
import cohere 
import os
#https://docs.cohere.com/docs/models#command

temperature=0.7
model="command-r7b-12-2024"

# Initialize Cohere client
co = cohere.Client(api_key=os.environ["COHERE"])  # or put your key directly

def inference(prompt): 
    response = co.chat(
        message= prompt, 
        model = model, 
        temperature = temperature 
    )
    return response.text 

In [27]:
print(inference(prompt))

```json
{
  "top_bottlenecks": [compute_1455, compute_1456, compute_1457],
  "suggestions": [
    "Increase microbatches to reduce the overhead of data loading and processing.",
    "Overlap communication and computation stages to improve resource utilization.",
    "Remap stages to different GPUs to balance the workload and reduce potential bottlenecks."
  ]
}
```


## 4. What-If Validation of Suggestions

TODO: Use what-if harness to validate LLM suggestions

In [ ]:
# TODO: Load what-if harness from notebook 06
# TODO: Apply top-3 suggestions to test graphs
# TODO: Run GNN forward pass on counterfactually edited graphs
# TODO: Measure ΔT improvement for each suggestion
# TODO: Calculate suggestion accuracy metric (≥2 of 3 reduce ΔT)
pass

## 5. Recall@1 Evaluation

TODO: Evaluate if top suggestions match known bottlenecks

In [ ]:
# TODO: Identify known bottlenecks in test graphs (ground truth)
# TODO: Compare LLM top suggestion vs ground truth bottleneck
# TODO: Compute Recall@1 metric across ≥20 graphs
# TODO: Generate results table for paper
pass

## 6. Batch Processing & Results Compilation

TODO: Run GraphRAG loop on multiple graphs and compile results

In [ ]:
# TODO: Implement batch processing for ≥20 graphs
# TODO: Store all suggestions and validation results
# TODO: Compile summary statistics
# TODO: Generate analysis plots and tables
pass

## 7. Final Summary & Paper Results

TODO: Generate final results and summary for publication

In [ ]:
# TODO: Generate final results summary
# TODO: Create tables/figures for paper
# TODO: Document limitations and future work
# TODO: Verify all acceptance criteria met
pass

print("Sprint 6 GraphRAG Status:")
print("- Prompt template: TODO")
print("- LLM integration: TODO")
print("- Suggestion validation: TODO")
print("- Recall@1 evaluation: TODO")